[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/cv-internship-in-a-book/blob/main/notebooks/week10_explainable_ai_STARTER.ipynb)


# Week 10 -- Explainable AI — What Is the Model Looking At? (STARTER)
### The Computer Vision Internship | MediVision AI Lagos

**This week:** GradCAM + LIME + SHAP + Integrated Gradients — all taught through code, all applied to cancer

**Dataset:** Chest X-Ray + BreakHis + TCGA (XAI)

**STARTER notebook** -- your working environment. Complete each exercise before moving on.


In [ ]:
# -- Colab/Local Setup -- run this first in Colab, skip locally -------------
import os

if not os.path.exists('requirements.txt'):
    # Clone the full course repository
    !git clone https://github.com/internshipinabook/cv-internship-in-a-book.git cv-book
    %cd cv-book
    # Install all required packages (suppress verbose output with -q)
    !pip install -r requirements.txt -q

# Create working directories
os.makedirs('outputs',  exist_ok=True)   # charts, reports, predictions
os.makedirs('models',   exist_ok=True)   # trained model checkpoints
os.makedirs('datasets', exist_ok=True)   # raw downloaded data
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds --------------------------------------------------
# Setting seeds ensures every random operation produces the same result
# on every run. Essential for: comparing runs, debugging, sharing results.

import random    # Python built-in random
import numpy as np   # NumPy random (data loading, sklearn)
import torch     # PyTorch random (neural networks, dropout)

SEED = 42        # 42 is conventional -- any integer works

random.seed(SEED)           # fix Python random() calls
np.random.seed(SEED)        # fix np.random.* calls
torch.manual_seed(SEED)     # fix PyTorch CPU operations

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)  # fix GPU operations too

# Prefer deterministic cuDNN algorithms (may slow training ~5%)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Seeds fixed: {SEED} | Device: {device}')


In [ ]:
# -- Monday: GradCAM -- Which Region Drove the Prediction? -----------------
# GradCAM (Gradient-weighted Class Activation Mapping) answers:
#   'Which spatial region of this image most influenced the predicted class?'
#
# Algorithm:
#   1. Forward pass -> get class score for the target class
#   2. Compute gradient of class score w.r.t. last conv feature map
#   3. Global average pool the gradients -> one importance weight per channel
#   4. Weighted sum of feature maps using those weights
#   5. ReLU (keep only positive contributions)
#   6. Upsample to original image size

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

# Step 1: Specify the target layer
# We use the LAST convolutional layer (model.layer4[-1] for ResNet-18)
# WHY the last layer?
#   - Highest semantic content (most abstract features)
#   - Still has spatial dimensions (not yet globally pooled)
#   - Earlier layers have higher resolution but lower-level features
target_layers = [model.layer4[-1]]   # adjust if using a different backbone

# Step 2: Create GradCAM -- registers hooks on target_layers
cam = GradCAM(model=model, target_layers=target_layers)

# Step 3: Load and preprocess a chest X-ray
try:
    img_pil = Image.open('datasets/xray/sample_pneumonia.jpg').convert('RGB').resize((224,224))
except FileNotFoundError:
    img_pil = Image.fromarray((np.random.rand(224,224,3)*255).astype('uint8'))
    print('Using synthetic image -- replace with real chest X-ray')

# img_arr: float32 [0,1] HWC -- used for the overlay visualisation
img_arr = np.array(img_pil).astype(np.float32) / 255.0

# tensor: preprocessed CHW float32 with batch dim -- fed to the model
tensor = preprocess_xray(img_pil).unsqueeze(0).to(device)  # (1,3,224,224)

# Step 4: Define the target class
# ClassifierOutputTarget(idx): use gradients w.r.t. class at index idx
PNEUMONIA_IDX = 6   # check your CONDITIONS list for the correct index
targets = [ClassifierOutputTarget(PNEUMONIA_IDX)]

# Step 5: Generate the heatmap
# cam() runs forward pass, computes gradients, returns (1,224,224) float heatmap
grayscale_cam = cam(input_tensor=tensor, targets=targets)
heatmap = grayscale_cam[0]   # extract from batch -- shape (224,224)

# Step 6: Create the overlay
# show_cam_on_image: blend heatmap with original image
# img_arr must be float32 [0,1]; use_rgb=True for the red-blue colourmap
overlay = show_cam_on_image(img_arr, heatmap, use_rgb=True)

# Step 7: Visualise all three panels
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_arr, cmap='gray');   axes[0].set_title('Original X-ray')
axes[1].imshow(heatmap, cmap='hot');    axes[1].set_title('GradCAM Heatmap\n(bright = model attended here)')
axes[2].imshow(overlay);               axes[2].set_title('Overlay\n(where did the model look?)')
for ax in axes: ax.axis('off')  # hide tick marks -- irrelevant for images
plt.suptitle('GradCAM -- Pneumonia prediction')
plt.tight_layout()
plt.savefig('outputs/week10_gradcam.png', dpi=100, bbox_inches='tight')
plt.show()

# Step 8: Clinical interpretation
max_y, max_x = np.unravel_index(heatmap.argmax(), heatmap.shape)
region = f"{'lower' if max_y > 112 else 'upper'} {'left' if max_x < 112 else 'right'}"
print(f'Peak attention: {region} region')
print('NOTE: GradCAM shows WHERE the model looked -- not WHY it was correct.')
print('A sensible-looking heatmap is NOT validation. It is the start of validation.')


In [ ]:
# -- Tuesday: LIME -- Which Superpixels Changed the Decision? --------------
# LIME treats the model as a black box (no access to gradients).
# It segments the image into superpixels, randomly masks subsets,
# runs the model on each masked version, then fits a linear model:
#   'which superpixels, when removed, most change the prediction?'
#
# ADVANTAGE: works on ANY classifier
# DISADVANTAGE: slow (500+ forward passes), stochastic

from lime import lime_image
from skimage.segmentation import mark_boundaries  # visualise superpixel borders
import numpy as np, torch
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T

# Load a BreakHis malignant patch
try:
    img_arr = np.array(Image.open('datasets/breakhis/malignant/sample.png').convert('RGB'))
except FileNotFoundError:
    # Synthetic patch with H&E-like purple tint
    img_arr = np.random.randint(150, 255, (224, 224, 3), dtype='uint8')
    img_arr[:, :, 0] = np.random.randint(180, 220, (224, 224))  # lower red
    img_arr[:, :, 2] = np.random.randint(170, 230, (224, 224))  # higher blue (purple)

# LIME calls predict_fn with a list of HWC uint8 NumPy arrays
# It must return (N, num_classes) float probabilities
def predict_fn(images):
    # Build the preprocessing transform (same as used during training)
    preprocess = T.Compose([
        T.ToTensor(),   # HWC uint8 -> CHW float32 [0,1]
        T.Normalize([0.787, 0.621, 0.742], [0.148, 0.197, 0.139])  # BreakHis stats
    ])
    # Convert each LIME-perturbed image to a tensor and stack into a batch
    batch = torch.stack([preprocess(Image.fromarray(img)) for img in images]).to(device)
    with torch.no_grad():   # no gradients needed -- just reading predictions
        logits = model(batch)  # (N, 2) raw logits
    # softmax converts logits to probabilities summing to 1 per sample
    probs = torch.softmax(logits, dim=1)
    return probs.cpu().numpy()  # LIME requires NumPy array

# Create the LIME explainer
explainer = lime_image.LimeImageExplainer(random_state=SEED)

# Generate explanation -- num_samples=500 perturbed images
# More samples = more stable explanation, but slower
print('Generating LIME explanation (approx 30s on CPU)...')
explanation = explainer.explain_instance(
    img_arr,          # HWC uint8 image to explain
    predict_fn,       # black-box prediction function
    top_labels=1,     # explain only the top predicted class
    num_samples=500,  # number of masked variations to test
    hide_color=0,     # masked superpixels -> black (0)
    random_seed=SEED
)

top_class = explanation.top_labels[0]
print(f'Top predicted class: {top_class} ({"Malignant" if top_class==1 else "Benign"})')

# Extract superpixels supporting the prediction
# positive_only=True: only show superpixels that INCREASE the predicted class score
# num_features=5: show the 5 most influential superpixels
temp_img, mask = explanation.get_image_and_mask(
    top_class, positive_only=True, num_features=5, hide_rest=False)

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_arr)
axes[0].set_title('Original Malignant Patch')
axes[1].imshow(mark_boundaries(temp_img / 255.0, mask))
axes[1].set_title('LIME: top 5 superpixels\nsupporting the prediction')
for ax in axes: ax.axis('off')
plt.savefig('outputs/week10_lime.png', dpi=100, bbox_inches='tight')
plt.show()
print('Green outlines = superpixels that support the prediction.')
print('Do they contain nuclear pleomorphism or hyperchromasia?')
print('Run LIME 3 times -- if the same superpixels appear, the explanation is stable.')


## Learning Objectives

By the end of this week, you will be able to:

- Apply GradCAM to the chest X-ray model and identify which regions drive pathology predictions
- Apply LIME to the breast cancer model and identify which tissue superpixels matter most
- Apply SHAP DeepExplainer to the uterine cancer model and compute feature attributions
- Apply Integrated Gradients and compare it to GradCAM and LIME on the same image
- Write an XAI report for Nurse Folake that answers "what did the model see?"



---

## MONDAY -- "GradCAM — Which Region Activated the Prediction?"


GradCAM (Gradient-weighted Class Activation Mapping) computes the gradient of the class score with respect to the final convolutional feature map. Regions with large positive gradients are most important for the prediction. The result is a heatmap overlaid on the original image.


### Exercise 10.1 -- GradCAM on all domains

Apply GradCAM to one correct prediction and one incorrect prediction from each model (chest X-ray, BreakHis, TCGA). For each incorrect prediction: does the heatmap reveal why the model was wrong?


In [ ]:
# Exercise 10.1: GradCAM on all domains
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Monday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Monday: GradCAM — Which Region Activated the Prediction? (scaffold) --
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np
from PIL import Image

# Target: the final conv layer of ResNet-18
target_layer = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

# Load a chest X-ray predicted as Pneumonia
img = Image.open("sample_cxr.jpg").convert("RGB").resize((224,224))
img_arr = np.array(img) / 255.0
tensor  = preprocess(img).unsqueeze(0).to(device)

# Generate the heatmap
targets = [ClassifierOutputTarget(PNEUMONIA_CLASS_IDX)]
mask    = cam(input_tensor=tensor, targets=targets)
heatmap = show_cam_on_image(img_arr.astype(np.float32), mask[0], use_rgb=True)

# Display
import matplotlib.pyplot as plt
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(img); plt.title("Original X-ray"); plt.axis("off")
plt.subplot(1,3,2); plt.imshow(mask[0], cmap="hot"); plt.title("GradCAM heatmap"); plt.axis("off")
plt.subplot(1,3,3); plt.imshow(heatmap); plt.title("Overlay"); plt.axis("off")
plt.suptitle("GradCAM — Pneumonia prediction: where did the model look?")


### Monday Morning Moment

*Slack — Monday, 2:00pm.*

**Ngozi Eze-Okafor:** GradCAM is done for the chest X-ray model. The Pneumonia heatmap highlights the lower right lung field. That's... actually where consolidation appears in pneumonia?

**Dr. Kwame Asante:** It is. Lower lobe pneumonia is more common. The model appears to have learned that. But look at the Hernia prediction.

**Ngozi Eze-Okafor:** The heatmap for Hernia is highlighting the upper left region. Near the diaphragm?

**Dr. Kwame Asante:** Hiatal hernia would appear near the diaphragm. The model may be looking in the right place.

**Ngozi Eze-Okafor:** Or it may be looking at the diaphragm contour because that feature correlates with hernia in the training set — for the wrong reason.

**Dr. Kwame Asante:** You cannot tell from GradCAM alone. That is why we have four methods. Compare them.




---

## TUESDAY -- "LIME — Which Superpixels Changed the Decision?"


LIME (Local Interpretable Model-agnostic Explanations) perturbs the input image by masking superpixels (groups of pixels with similar colour/texture) and observes how the prediction changes. Superpixels whose removal most decreases the predicted probability are the most important. LIME is slower than GradCAM (requires many forward passes) but model-agnostic.


### Exercise 10.2 -- LIME superpixel count sensitivity

Run LIME with 3, 5, 10, and 20 top features. Display all four. At what number do the highlighted regions stabilise? What does instability with few features suggest about the explanation quality?


In [ ]:
# Exercise 10.2: LIME superpixel count sensitivity
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Tuesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Tuesday: LIME — Which Superpixels Changed the Decision? (scaffold) --
import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

explainer = lime_image.LimeImageExplainer()

# Load a BreakHis malignant patch
img_arr = np.array(Image.open("malignant_patch.png").convert("RGB"))

def predict_fn(images):
    tensors = torch.stack([preprocess(Image.fromarray(img)) for img in images])
    with torch.no_grad():
        logits = model(tensors.to(device))
    return torch.softmax(logits, dim=1).cpu().numpy()

explanation = explainer.explain_instance(
    img_arr, predict_fn, top_labels=1, num_samples=500, hide_color=0
)
# Get superpixels that support the malignant prediction
temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0], positive_only=True, num_features=5
)
plt.imshow(mark_boundaries(temp/255., mask))
plt.title("LIME: superpixels supporting 'Malignant' prediction")



---

## WEDNESDAY -- "SHAP — How Much Did Each Feature Contribute?"


SHAP (SHapley Additive exPlanations) computes the contribution of each input feature to the model output using game-theoretic Shapley values. DeepExplainer is the SHAP method for deep neural networks — it uses a background distribution of images as the baseline.


### Exercise 10.3 -- SHAP vs GradCAM agreement

For 10 images from BreakHis, compute: do SHAP and GradCAM highlight the same region (top-50% activation)? Use IoU (Intersection over Union) of the top-activation masks. Report mean IoU. Is agreement higher on correct or incorrect predictions?


In [ ]:
# Exercise 10.3: SHAP vs GradCAM agreement
# -------------------------------------------------------
# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Wednesday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Wednesday: SHAP — How Much Did Each Feature Contribute? (scaffold) --
import shap
import torch

# Load background images (50 random training images)
background = torch.stack([preprocess(Image.open(p).convert("RGB"))
                          for p in train_paths[:50]]).to(device)

# Create SHAP explainer
explainer = shap.DeepExplainer(model, background)

# Explain one uterine cancer prediction
test_img = preprocess(Image.open("cancer_patch.png").convert("RGB")).unsqueeze(0).to(device)
shap_values = explainer.shap_values(test_img)

# shap_values[class_idx] has shape (1, C, H, W) — one attribution per pixel per channel
# Aggregate across channels for visualisation
pixel_importance = np.abs(shap_values[CANCER_CLASS]).mean(axis=1)[0]  # (H, W)
plt.imshow(pixel_importance, cmap="hot")
plt.colorbar(label="SHAP attribution magnitude")
plt.title("SHAP: pixel-level feature importance for 'Cancer' prediction")



---

## THURSDAY -- "Integrated Gradients and the XAI Comparison"


Integrated Gradients (Sundararajan et al., 2017) computes the integral of gradients along a straight path from a baseline input (usually a black image) to the actual input. Unlike GradCAM, it provides pixel-level attributions without requiring access to intermediate feature maps — it works on the input itself.


### STOP AND TRACE

Before running: what does "gradient of the class score with respect to the feature map" mean?

targets = [ClassifierOutputTarget(PNEUMONIA_CLASS_IDX)]
mask = cam(input_tensor=tensor, targets=targets)

Trace: (1) what is ClassifierOutputTarget computing? (2) which layer's feature map is being used? (3) why use the last conv layer rather than an earlier one?
Why this line exists: the last convolutional layer has the highest-level semantic features but still retains spatial information. Earlier layers have higher resolution but lower-level features. GradCAM is a tradeoff between semantic content and spatial precision.


In [ ]:
# Exercise 10.4: STOP AND TRACE — GradCAM gradient flow
# -------------------------------------------------------
# STOP AND TRACE: before running, write your expected output as a comment.
# Then run the cell and compare.

# YOUR PREDICTION:
# Expected output: ???

# YOUR CODE HERE
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the bottom of this notebook.


**Thursday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Thursday: Integrated Gradients and the XAI Comparison (scaffold) --
from captum.attr import IntegratedGradients
import torch

ig = IntegratedGradients(model)

test_img = preprocess(Image.open("cancer_patch.png").convert("RGB")).unsqueeze(0).to(device)
test_img.requires_grad = True

# Baseline: black image
baseline = torch.zeros_like(test_img)

attributions, delta = ig.attribute(
    test_img, baseline, target=CANCER_CLASS,
    return_convergence_delta=True, n_steps=200
)
print(f"Convergence delta (should be small): {delta.item():.6f}")

attr_map = attributions[0].permute(1,2,0).detach().cpu().numpy()
attr_map = np.abs(attr_map).mean(axis=2)  # aggregate channels
plt.imshow(attr_map, cmap="hot")
plt.title("Integrated Gradients: attribution map for cancer prediction")



---

## FRIDAY -- "The Friday Build: The XAI Report for Nurse Folake"


Take one chest X-ray predicted as Pneumonia. Generate all four explanations: GradCAM, LIME, SHAP, Integrated Gradients. Display them side by side. Write a one-page clinical XAI report answering: "Here is what the model saw." Do all four methods agree? If not, which disagreement is clinically significant? Which explanation would Nurse Folake find most actionable?


**Friday code scaffold** (read and run, then adapt in the exercise cells above):


In [ ]:
# -- Friday: The Friday Build: The XAI Report for Nurse Folake (scaffold) --
# XAI comparison report:
# 1. Four panels: GradCAM | LIME | SHAP | IntGrad — same image
# 2. Do they agree on the region? Yes/No/Partially
# 3. Which region does each method highlight?
# 4. Clinical interpretation: is the highlighted region the correct region?
# 5. Which method would a radiologist find most legible? Why?
# 6. "The model appears to have based this Pneumonia prediction on X"


### Friday Workplace Moment

*MediVision AI — Friday, 11:00am.*

**Nurse Folake Balogun:** These four images all look different. Which one is telling me where the pneumonia is?

**Ngozi Eze-Okafor:** All four are. GradCAM and LIME agree that the lower right lung field is the key region. SHAP is more diffuse — it highlights a larger area including the right hilum. Integrated Gradients is the most precise — it highlights a roughly 4×4cm region in the right lower lobe.

**Nurse Folake Balogun:** Show me the one that looks most like what a radiologist would circle.

**Dr. Kwame Asante:** That is GradCAM or Integrated Gradients — both produce spatially coherent regions. LIME superpixels look like patches, not anatomical regions.

**Nurse Folake Balogun:** Can I use this to explain a decision to a patient?

**Ngozi Eze-Okafor:** With important caveats. GradCAM shows where the model looked. It does not confirm the model looked for the right reason. The image should be verified by a radiologist before it is shown to a patient.

**Dr. Kwame Asante:** Write that as the first sentence of the clinical limitations section.



## YOUR WEEK 10 CHECKLIST

Check each item before moving to the next week.
If you cannot explain an item without notes, revisit that section.

- [ ] I can explain in plain English what GradCAM, LIME, SHAP, and Integrated Gradients each measure — without equations.
- [ ] I applied all four methods to the same image and can state whether they agree on the critical region.
- [ ] I know why a GradCAM that looks sensible does not validate the model's reasoning.
- [ ] My XAI report for Nurse Folake states what the model saw, not just what the method computed.
- [ ] I can compute IoU between two attribution masks and interpret the result.


## Challenge Task

> **Core path:** Implement GradCAM++ (an improvement on GradCAM that uses second-order gradients). Compare GradCAM and GradCAM++ on 10 images from BreakHis. Which produces sharper localisation? Quantify using the bounding box overlap with the annotated tumour region.

> **Advanced path:** Find one image where GradCAM and Integrated Gradients disagree strongly (IoU < 0.2). Investigate why. Is the model attending to different features, or is the disagreement an artefact of the explanation method?


## Common Mistakes

**Treating GradCAM as validation of model correctness:** A heatmap that highlights the correct anatomical region confirms where the model looked, not why. The model may look at the right region for the wrong reason.

**Using LIME with too few samples:** LIME requires many perturbations (500+) to produce stable explanations. With 100 samples, the explanation may change each time you run it.

**Comparing XAI methods without a ground truth:** IoU between GradCAM and LIME measures agreement between methods, not correctness. Ground truth requires expert annotation of the clinically relevant region.



---

## Get the Full Book

This notebook is part of **The Computer Vision Internship** -- Book 3 of 9, InternshipInABook(tm).

Covers: natural images, medical X-rays, breast & uterine cancer, video, GradCAM/LIME/SHAP/Integrated Gradients, and fairness audits.

Available at [internshipinabook.com](https://internshipinabook.com)
